# **NORMALIZACIÓN DE VARIABLES MULTIVALUE (INTEREST_TAGS)**

En esta sección se transforma la columna `interest_tags`, que contiene múltiples intereses separados por comas, a un formato estructurado.

Este proceso permite:

- Analizar la distribución real de intereses
- Facilitar su uso en visualizaciones
- Mejorar la segmentación de usuarios

Se genera un dataset en formato largo, donde cada fila representa un único interés asociado a un usuario.

In [10]:
import pandas as pd
import numpy as np


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

#### **1. CARGA DE LOS DATASETS**

- Importar el dataset previamente limpiado y transformado durante la fase de data cleaning, que servirá como base para aplicar transformaciones adicionales orientadas a facilitar el análisis y la visualización.
- Tras la carga, se realiza una verificación básica para confirmar la correcta lectura del archivo, sus dimensiones y su estructura, garantizando que el dataset está preparado para la generación de datasets derivados utilizados en fases analíticas posteriores.

In [11]:
path_df1 = "../data/processed/dating_app_behavior_clean.csv"

df1 = pd.read_csv(path_df1, index_col=0)
df1 = df1.reset_index().rename(columns={"index": "gender"})
df1["user_id"] = df1.index

print(f"Dataset 1: {df1.shape[0]} filas × {df1.shape[1]} columnas")
display(df1.head())

Dataset 1: 50000 filas × 20 columnas


,gender,sexual_orientation,location_type,income_bracket,education_level,interest_tags,app_usage_time_min,app_usage_time_label,swipe_right_ratio,swipe_right_label,likes_received,mutual_matches,profile_pics_count,bio_length,message_sent_count,emoji_usage_rate,last_active_hour,swipe_time_of_day,match_outcome,user_id
0,Prefer Not To Say,Gay,Urban,High,Bachelor’S,"Fitness, Politics, Traveling",52,Moderate,0.60,Optimistic,173,23,4,44,75,0.36,13,Early Morning,Mutual Match,0
1,Male,Bisexual,Suburban,Upper-Middle,No Formal Education,"Languages, Fashion, Parenting",279,Extreme User,0.56,Optimistic,107,7,3,301,35,0.42,0,Morning,Chat Ignored,1
2,Non-Binary,Pansexual,Suburban,Low,Master’S,"Movies, Reading, Diy",49,Moderate,0.41,Optimistic,91,27,2,309,33,0.41,1,After Midnight,Date Happened,2
3,Genderfluid,Gay,Metro,Very Low,Postdoc,"Coding, Podcasts, History",185,Extreme User,0.32,Balanced,147,6,5,35,5,0.07,21,Morning,No Action,3
4,Male,Bisexual,Urban,Middle,Bachelor’S,"Clubbing, Podcasts, Cars",83,High,0.32,Balanced,94,11,1,343,34,0.11,22,After Midnight,One-Sided Like,4


#### **2. NORMALIZACIÓN DE LA VARIABLE DE INTERESES**

En esta sección se define una función para transformar la variable `interest_tags`, que contiene múltiples intereses por usuario en un único campo separado por comas, a un formato estructurado adecuado para su análisis.

Este proceso permite:

- Separar los intereses individuales asociados a cada usuario.
- Generar un dataset en formato largo, donde cada fila representa un único interés.
- Facilitar el análisis de la distribución de intereses y su uso en visualizaciones.

La transformación se realiza sobre una copia del dataset original, preservando su estructura principal y garantizando la integridad de las métricas a nivel de usuario.

In [12]:
def normalize_interests(df, column="interest_tags"):
    """
    Normaliza la columna de intereses a formato largo (una fila por interés).

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset original de usuarios
    column : str
        Nombre de la columna que contiene los intereses separados por comas

    Returns
    -------
    pandas.DataFrame
        Dataset en formato largo con una fila por interés
    """

    df_interests = df.copy()

    df_interests[column] = (
        df_interests[column]
        .astype(str)
        .str.split(",")
    )

    df_interests = df_interests.explode(column)

    df_interests[column] = (
        df_interests[column]
        .str.strip()
        .str.lower()
    )

    return df_interests

In [13]:
df_interests = normalize_interests(df1)
df_interests.head(15)

,gender,sexual_orientation,location_type,income_bracket,education_level,interest_tags,app_usage_time_min,app_usage_time_label,swipe_right_ratio,swipe_right_label,likes_received,mutual_matches,profile_pics_count,bio_length,message_sent_count,emoji_usage_rate,last_active_hour,swipe_time_of_day,match_outcome,user_id
0,Prefer Not To Say,Gay,Urban,High,Bachelor’S,fitness,52,Moderate,0.60,Optimistic,173,23,4,44,75,0.36,13,Early Morning,Mutual Match,0
0,Prefer Not To Say,Gay,Urban,High,Bachelor’S,politics,52,Moderate,0.60,Optimistic,173,23,4,44,75,0.36,13,Early Morning,Mutual Match,0
0,Prefer Not To Say,Gay,Urban,High,Bachelor’S,traveling,52,Moderate,0.60,Optimistic,173,23,4,44,75,0.36,13,Early Morning,Mutual Match,0
1,Male,Bisexual,Suburban,Upper-Middle,No Formal Education,languages,279,Extreme User,0.56,Optimistic,107,7,3,301,35,0.42,0,Morning,Chat Ignored,1
1,Male,Bisexual,Suburban,Upper-Middle,No Formal Education,fashion,279,Extreme User,0.56,Optimistic,107,7,3,301,35,0.42,0,Morning,Chat Ignored,1
1,Male,Bisexual,Suburban,Upper-Middle,No Formal Education,parenting,279,Extreme User,0.56,Optimistic,107,7,3,301,35,0.42,0,Morning,Chat Ignored,1
2,Non-Binary,Pansexual,Suburban,Low,Master’S,movies,49,Moderate,0.41,Optimistic,91,27,2,309,33,0.41,1,After Midnight,Date Happened,2
2,Non-Binary,Pansexual,Suburban,Low,Master’S,reading,49,Moderate,0.41,Optimistic,91,27,2,309,33,0.41,1,After Midnight,Date Happened,2
2,Non-Binary,Pansexual,Suburban,Low,Master’S,diy,49,Moderate,0.41,Optimistic,91,27,2,309,33,0.41,1,After Midnight,Date Happened,2
3,Genderfluid,Gay,Metro,Very Low,Postdoc,coding,185,Extreme User,0.32,Balanced,147,6,5,35,5,0.07,21,Morning,No Action,3


#### **3. EXPORTACIÓN DEL DATASET DE INTERESES NORMALIZADO**

- En esta sección se exporta el dataset de intereses previamente normalizado a la carpeta `data/processed/`.
- Este dataset se encuentra en formato largo, donde cada fila representa un interés individual asociado a un usuario, lo que facilita su uso en análisis posteriores y visualizaciones.
- La exportación separada de este dataset permite preservar la integridad del dataset principal de usuarios, evitando la duplicación de registros y garantizando la consistencia del modelo analítico.

In [14]:
# Definir ruta de exportación
output_path = "../data/processed/users_interests_long.csv"

# Exportar dataset de intereses normalizado
df_interests.to_csv(output_path, index=False)

# Confirmación
print(f"Dataset de intereses exportado correctamente en: {output_path}")

Dataset de intereses exportado correctamente en: ../data/processed/users_interests_long.csv
